# inplace-param-update — faded example 2: Add the no_grad guard around the update

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `inplace-param-update`. The last cell reports your progress on the `PyTorch: In-place param update` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: In-place param update` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inplace-param-update`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inplace-param-update"
DD_SUBTOPIC = "PyTorch: In-place param update"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Parameter updates must not be recorded by autograd, or the graph grows every step and `requires_grad` propagates into the weights. Wrapping the in-place `p -= lr * p.grad` in a `torch.no_grad()` block keeps the update untracked while still mutating the model's storage.

## Faded exercise 2

Complete `update_in_no_grad(model, lr)`. After a backward pass it should update each parameter in place under a no_grad context, then zero the grads. Fill in the context-manager line that makes the update untracked.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn
import contextlib

t.manual_seed(4)

def update_in_no_grad(model, lr):
    ctx = t.no_grad()
    with ctx:
        for p in model.parameters():
            p -= lr * p.grad
            p.grad.zero_()
    return model


def _test():
    lin = nn.Linear(4, 1)
    x = t.randn(6, 4)
    loss = (lin(x) ** 2).sum()
    loss.backward()
    grads_before = [p.grad.detach().clone() for p in lin.parameters()]
    vals_before = [p.detach().clone() for p in lin.parameters()]
    ptrs = [p.data_ptr() for p in lin.parameters()]
    update_in_no_grad(lin, lr=0.1)
    for p, vb, gb, ptr in zip(lin.parameters(), vals_before, grads_before, ptrs):
        # independent truth: new value = old - 0.1 * (grad recorded before)
        assert t.allclose(p.detach(), vb - 0.1 * gb)
        assert p.data_ptr() == ptr, 'in-place update must keep storage'
        # params must remain leaves with requires_grad still True and no tracked history
        assert p.requires_grad and p.grad_fn is None
        # grads were zeroed
        assert t.allclose(p.grad, t.zeros_like(p.grad))


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn
import contextlib

t.manual_seed(4)

def update_in_no_grad(model, lr):
    ctx = t.no_grad()
    with ctx:
        for p in model.parameters():
            p -= lr * p.grad
            p.grad.zero_()
    return model
```
</details>